## Imports

In [ ]:
import tensorflow.keras as keras
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from keras.models import Sequential
from keras.layers import Dense, Input, Flatten, Conv2D, MaxPooling2D
from keras.utils import plot_model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [ ]:
tf.config.list_physical_devices('GPU')


## Overview of Keras
The following are the common steps you want to take when fitting a neural network using keras:
1. Load data
2. Define keras model
3. Compile model
4. Fit model
5. Evaluate model
6. Use model for prediction

## References

- [Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow, 3rd Edition]( https://learning.oreilly.com/library/view/hands-on-machine-learning/9781098125967/ )
  - Accompanying [GitHub repo]( https://github.com/ageron/handson-ml3 )


## Example 1 - Diabetes Prediction

### Load data

In [ ]:
url = 'https://ddc-datascience.s3.amazonaws.com/pima-indians-diabetes.csv'
url


In [ ]:
col_names = ['pregnant', 'glucose', 'bp', 'skin', 'insulin', 'bmi', 'pedigree', 'age', 'label']
diab_data = pd.read_csv( url, header = None, names = col_names)
diab_data.head()


In [ ]:
diab_data.shape


In [ ]:
diab_data.describe().transpose()


In [ ]:
diab_data.info()


In [ ]:
# Separate predictors and response
diab_predictors = diab_data.drop('label', axis = 1).copy()
diab_response = diab_data['label'].copy()


In [ ]:
# Split the data up in train and test sets
X_train, X_test, y_train, y_test = train_test_split(diab_predictors, diab_response, test_size=0.25, random_state=42)


In [ ]:
X_train.shape


In [ ]:
# Define the scaler
scaler = StandardScaler().fit(X_train)

# Scale the train set
X_train = scaler.transform(X_train)

# Scale the test set
X_test = scaler.transform(X_test)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
diab_data["label"].value_counts( dropna = False )


### Define keras model

Models in keras are produced using a sequence of layers. We sequentially add layers until we are happy with our model.

In [ ]:
model = Sequential()

Define input layer where the number of nodes is the number of features in the data set.

In [ ]:
model.add(
  Input(shape=(X_train.shape[1],))
)


Define first hidden layer - need to specify activation function. We are using a Dense layer, which is a fully connected layer.

In [ ]:
model.add(
  Dense(
    name = "Hidden",
    units = 8,
    activation = 'relu',
  )
)

Define output layer

In [ ]:
model.add(
  Dense(
    name = "Output",
    units = 1,
    activation = 'sigmoid',
  )
)

### Compile model

In [ ]:
# Using binary cross entropy loss function and accuracy as metric since we are doing a binary prediction
model.compile(
  optimizer = 'adam',
  loss = 'binary_crossentropy',
  metrics = ['accuracy']
)

### Fit model

In [ ]:
# Fit model using training data
model.fit(X_train, y_train, epochs=12) ;


In [ ]:
model.metrics[1].metrics


In [ ]:
model.summary()


In [ ]:
plot_model(model)


### Evaluate model

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print(accuracy)


### Use model for prediction

In [ ]:
predictions = model.predict(X_test)


In [ ]:
predictions.shape


In [ ]:
predictions[1]


Create a threshold.

In [ ]:
filter = ( predictions > 0.5 )
filter[:10]


In [ ]:
# Get class prediction by converting boolean to integers
class_pred = filter.astype("int32")
class_pred[1]

In [ ]:
# Compare to true value
y_test.iloc[1]

In [ ]:
predictions[:10]

In [ ]:
predictions.shape

In [ ]:
results = pd.DataFrame( {
  "Truth": y_test,
  "Prediction": predictions[:,0],
  "Class": class_pred[:,0],
} )

In [ ]:
results.shape

In [ ]:
results.head()

First five mismatches.

In [ ]:
results.query("Truth != Class").head()

Same first five mismatches using a filter.

In [ ]:
filter_mismatch = ( results["Truth"] != results["Class"] )
results[filter_mismatch].head()

In [ ]:
filter_mismatch.sum()

In [ ]:
1-filter_mismatch.mean()

## Example 2 - Number Recognition

### Load data

We will be looking at an example dataset from keras called MNIST.

In [ ]:
mnist = tf.keras.datasets.mnist
(x_train, y_train),(x_test, y_test) = mnist.load_data() # Automatically shuffles data into random training and testing sets

In [ ]:
(
  type(x_train),
  type(y_train),
  type(x_test),
  type(y_test)
)


In [ ]:
(
  x_train.shape,
  y_train.shape,
  x_test.shape,
  y_test.shape
)


In [ ]:
pd.Series(list(y_train)+list(y_test)).value_counts( normalize=True ) * 100


In [ ]:
x_train.shape # 60,000 images that are 28 x 28 pixels

In [ ]:
# shape of a single image
x_train.shape[1:]

The first image as a matrix of pixel intensities.

In [ ]:
x_train[0]

The 11th row of the first image.

In [ ]:
x_train[0][10]

The pixel intensity of the 10th column of the 11th row of the first image.


In [ ]:
x_train[0][10][9]

In [ ]:
# Visualize one of the training samples
plt.imshow( x_train[0], cmap = plt.cm.gray_r )
plt.show()

In [ ]:
# See the response for that sample
y_train[0]

### Define keras model

First, we will normalize our data values to fall between 0 and 1.

In [ ]:
# View before normalization
x_train[0][10]

In [ ]:
x_train = tf.keras.utils.normalize(x_train, axis=1)
x_test = tf.keras.utils.normalize(x_test, axis=1)

In [ ]:
# View after normalization
x_train[0][10]

In [ ]:
plt.imshow(x_train[0], cmap=plt.cm.gray_r)
plt.show()

Start the model as a feed forward (sequential) model.

In [ ]:
28*28

In [ ]:
model = Sequential()

# Flatten input data into a 1D structure
model.add(Flatten())

# Define first hidden layers
model.add(
  Dense(
    name = "hidden1",
    units = 144,
    activation = 'relu' ,
  )
)

# Add second hidden layer
model.add(
  Dense(
    name = "hidden2",
    units = 144,
    activation = 'relu' ,
  )
)

# Define output layer
model.add(
  Dense(
    name = "output",
    units = 10,
    activation = 'softmax' ,
  )
)

### Compile model

In [ ]:
model.compile(
    optimizer = 'adam',
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy'],
)

### Fit model

In [ ]:
model.fit(x_train, y_train, epochs=8) ;

In [ ]:
model.summary()

In [ ]:
784*144, 784*144+144

In [ ]:
144*144, 144*144+144

In [ ]:
144*10, 144*10+10

In [ ]:
(784*144+144) + ( 144*144+144) + ( 144*10+10)

In [ ]:
plot_model(model)

### Evaluate model

In [ ]:
val_loss, val_acc = model.evaluate(x_test, y_test)
val_acc

### Use model for prediction

In [ ]:
predictions = model.predict(x_test)


In [ ]:
i=257
# i=4348
i

In [ ]:
print(predictions[i])

In [ ]:
[ print(f"{k}: {x:.2f}") for k, x in enumerate(predictions[i]*100) ]


In [ ]:
x_test.shape

In [ ]:
predictions.shape

In [ ]:
print(np.argmax(predictions[i]))

In [ ]:
y_test[i]

In [ ]:
plt.imshow(x_test[i], cmap=plt.cm.gray_r)
plt.show()